# US Market Brief

Notebook นี้ดึงข้อมูลตลาดหุ้นอเมริกาสำหรับรายงานสรุปตลาดยามเช้า (เวลาไทย) แล้วบันทึกผลเป็นไฟล์ JSON

> **สถานะ:** กำลังปรับจากตลาดหุ้นไทยเป็นตลาดอเมริกา ส่วน SET/TFEX ถูกตัดออกแล้ว ข้อมูลตลาด US จะเพิ่มทีละขั้น

## วิธีใช้งาน

1. รัน Cell 1 และ 1B เฉพาะครั้งแรก (ติดตั้งไลบรารีและ Chromium)
2. แก้ `REPORT_DATE` ใน Cell 2 เฉพาะเมื่อต้องการทดสอบวันอื่น
3. รันทุก Cell จากบนลงล่าง แล้วเปิดไฟล์ JSON ที่ได้ ตรวจว่า `errors` เป็นรายการว่าง

ค่าที่แก้ได้อย่างปลอดภัย: `REPORT_DATE`, `CUTOFF_TIME`, `TIMEOUT_SECONDS`

In [ ]:
# Cell 1 — รายการไลบรารีที่ต้องติดตั้ง (ใช้คำสั่งด้านล่างด้วยตนเองเฉพาะครั้งแรก)
#%pip install -q requests beautifulsoup4 pandas lxml playwright nbformat nbclient yfinance
#%pip install "pandas<3"

In [ ]:
# Cell 1B — ติดตั้ง Chromium เฉพาะครั้งแรก โดยเปลี่ยน INSTALL_CHROMIUM เป็น True แล้วรัน Cell นี้
# หลังติดตั้งสำเร็จให้เปลี่ยนกลับเป็น False
import sys
import subprocess
INSTALL_CHROMIUM = False
if INSTALL_CHROMIUM:
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)

In [ ]:
# Cell 2 — นำเข้าไลบรารีและกำหนดค่า เปลี่ยน REPORT_DATE เมื่อต้องการทดสอบวันอื่น
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from io import StringIO
import os
import json
from pathlib import Path
from urllib.parse import urljoin
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

BANGKOK = ZoneInfo("Asia/Bangkok")
NEW_YORK = ZoneInfo("America/New_York")  # เขตเวลาของตลาดหุ้น US (ปรับ Daylight Saving ให้อัตโนมัติ)
REPORT_DATE = os.getenv("DAILY_FOCUS_REPORT_DATE") or None  # Notebook หลักส่งวันที่ผ่าน environment; หากไม่มีให้ใช้วันนี้
CUTOFF_TIME = "08:30:00"         # เวลาประเทศไทยที่ใช้เป็นจุดตัดข้อมูล
PAST_DAYS = 3
FUTURE_DAYS = 7
TIMEOUT_SECONDS = 25
OUTPUT_DIR = Path("web_scraping_test_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

report_day = datetime.strptime(REPORT_DATE, "%Y-%m-%d").date() if REPORT_DATE else datetime.now(BANGKOK).date()
cutoff = datetime.combine(report_day, datetime.strptime(CUTOFF_TIME, "%H:%M:%S").time(), tzinfo=BANGKOK)
calendar_start = report_day - timedelta(days=PAST_DAYS)
calendar_end = report_day + timedelta(days=FUTURE_DAYS)
print("วันที่จัดทำรายงาน:", report_day, "| เวลาตัดข้อมูล:", cutoff.isoformat())
print("เวลาตัดข้อมูลเมื่อเทียบเป็นเวลานิวยอร์ก:", cutoff.astimezone(NEW_YORK).isoformat())
print("ช่วงวันที่ที่ขอจาก Forex Factory:", calendar_start, "ถึง", calendar_end)

In [ ]:
# Cell 3 — ฟังก์ชันกลางสำหรับขอหน้าเว็บด้วย requests
# หากเกิดข้อผิดพลาดให้เก็บสถานะจริงไว้ และไม่สร้างข้อมูลขึ้นเอง
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; DailyFocusEducationalTest/1.0)",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.8",
})
statuses = []  # บันทึกผลการขอแต่ละหน้า ใช้ดูสรุปภายหลัง

def get_page(label, url, params=None):
    try:
        response = session.get(url, params=params, timeout=TIMEOUT_SECONDS)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        title = soup.title.get_text(" ", strip=True) if soup.title else ""
        return {"label": label, "ok": True, "status": response.status_code,
                "url": response.url, "html": response.text, "soup": soup,
                "title": title, "error": ""}
    except requests.RequestException as exc:
        status = getattr(getattr(exc, "response", None), "status_code", None)
        return {"label": label, "ok": False, "status": status,
                "url": url, "html": "", "soup": None, "title": "", "error": str(exc)[:250]}

def page_tables(page, limit=3):
    if not page["ok"]:
        return []
    try:
        return pd.read_html(StringIO(page["html"]))[:limit]
    except (ValueError, ImportError):
        return []

def show_page(label, page, extracted, note):
    statuses.append({"page": label, "http_status": page["status"],
                     "extracted_rows": len(extracted), "url": page["url"],
                     "note": note if page["ok"] else page["error"]})
    print(label, "| สถานะ HTTP:", page["status"], "| จำนวนแถวที่ดึงได้:", len(extracted))
    print("URL:", page["url"])
    if page["title"]:
        print("ชื่อหน้าเว็บ:", page["title"])
    if note or page["error"]:
        print("หมายเหตุ:", note or page["error"])
    if extracted:
        display(pd.DataFrame(extracted))
    else:
        tables = page_tables(page)
        print("จำนวนตาราง HTML ที่พบ (ตรวจสูงสุด 3 ตารางแรก):", len(tables))
        for table in tables:
            display(table.head(8))

In [ ]:
# Cell 3B — เปิดหน้าเว็บที่ต้องใช้ JavaScript ด้วย Chromium ผ่าน Python process แยก
# วิธีนี้เลี่ยงข้อผิดพลาด NotImplementedError ของ Playwright บน Windows Jupyter
# เรียกใช้ด้วย: page = await get_rendered_page("ชื่อ", "https://...")
import asyncio

_BROWSER_SCRIPT = r'''import json, sys
from playwright.sync_api import sync_playwright

url = sys.argv[1]
timeout_ms = int(sys.argv[2])
wait_ms = int(sys.argv[3])

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=True)
    try:
        context = browser.new_context(locale="en-US", timezone_id="Asia/Bangkok")
        page = context.new_page()
        response = page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
        page.wait_for_timeout(wait_ms)
        print(json.dumps({"status": response.status if response else None,
                          "ok": bool(response and response.ok),
                          "url": page.url, "html": page.content()}, ensure_ascii=True))
    finally:
        browser.close()
'''

async def get_rendered_page(label, url, wait_seconds=3):
    try:
        completed = await asyncio.to_thread(
            subprocess.run,
            [sys.executable, "-c", _BROWSER_SCRIPT, url,
             str(TIMEOUT_SECONDS * 1000), str(int(wait_seconds * 1000))],
            capture_output=True, text=True, encoding="utf-8",
            timeout=TIMEOUT_SECONDS + 30,
        )
        if completed.returncode:
            error = (completed.stderr or completed.stdout).strip()[-900:]
            raise RuntimeError(error or f"โปรเซสเบราว์เซอร์หยุดทำงานด้วยรหัส {completed.returncode}")
        result = json.loads(completed.stdout)
        soup = BeautifulSoup(result["html"], "html.parser")
        return {"label": label, "ok": result["ok"],
                "status": result["status"], "url": result["url"],
                "html": result["html"], "soup": soup,
                "title": soup.title.get_text(" ", strip=True) if soup.title else "",
                "error": ""}
    except Exception as exc:
        return {"label": label, "ok": False, "status": None, "url": url,
                "html": "", "soup": None, "title": "",
                "error": f"{type(exc).__name__}: {str(exc)[:900]}"}

In [ ]:
# Cell 3C — ฟังก์ชันช่วยทำความสะอาดข้อความและแปลงตัวเลข
from typing import Any

NUMBER_PATTERN = r"[+\-−]?\d[\d,]*(?:\.\d+)?"
PERCENT_PATTERN = rf"({NUMBER_PATTERN})\s*%"
PLACEHOLDERS = {"", "-", "--", "–", "—", "n/a", "na", "null", "none"}


def clean(value: Any) -> str:
    """ยุบช่องว่างซ้ำให้เหลือช่องเดียว โดยไม่เปลี่ยนค่าที่แสดง"""
    return re.sub(r"\s+", " ", str(value or "")).strip()


def is_real_value(value: Any) -> bool:
    """คืน False ถ้าเป็นค่าว่างหรือ placeholder เช่น '-', 'N/A'"""
    return clean(value).lower() not in PLACEHOLDERS


def first_match(text: str, pattern: str, flags: int = re.IGNORECASE) -> str | None:
    match = re.search(pattern, text, flags)
    if not match:
        return None
    value = clean(match.group(1))
    return value if is_real_value(value) else None


def numeric_value(text: str) -> float | None:
    """แปลงตัวเลขที่แสดง เช่น '1,234.5' หรือ '-0.8%' เป็น float"""
    match = re.search(NUMBER_PATTERN, clean(text).replace("−", "-"))
    if not match:
        return None
    try:
        return float(match.group(0).replace(",", ""))
    except ValueError:
        return None

## ดึงข้อมูลตลาด US

### ดัชนีหลัก

ดึงราคาปิดรายวันของ S&P 500, Nasdaq, Dow Jones และ Russell 2000 จาก Yahoo Finance แล้วเลือกวันซื้อขายล่าสุดที่ตลาดปิดก่อนเวลาตัดข้อมูล (`cutoff`) จากนั้นคำนวณการเปลี่ยนแปลงเทียบกับวันซื้อขายก่อนหน้า

- **Cell 4a:** นิยามฟังก์ชัน (ส่วนที่จะย้ายไปเป็นไฟล์ `.py`)
- **Cell 4b:** เรียกใช้และแสดงผล
- **Cell 4c:** ตรวจ logic การเลือกวันด้วยข้อมูลสมมติ โดยไม่ต้องใช้อินเทอร์เน็ต

*(ขั้นถัดไปจะเพิ่ม: Sector, ตัวชี้วัดตลาด, Futures)*

In [ ]:
# Cell 4a — นิยามฟังก์ชันดึงดัชนีหลัก (Cell นี้มีแต่ def และค่าคงที่ ตอนแยกเป็น .py จะย้ายไปทั้ง Cell)
# หลักการ: ฟังก์ชันรับค่าที่ต้องใช้ผ่านพารามิเตอร์ทั้งหมด ไม่แอบอ่านตัวแปรจาก Cell อื่น
from collections.abc import Callable
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import yfinance as yf

US_MARKET_TZ = ZoneInfo("America/New_York")
US_MARKET_CLOSE = datetime.strptime("16:00", "%H:%M").time()  # เวลาปิดตลาด ตามเวลานิวยอร์ก
HISTORY_DAYS = 14  # ขอย้อนหลังเผื่อวันหยุดยาว ต้องมีวันซื้อขายอย่างน้อย 2 วันจึงคำนวณการเปลี่ยนแปลงได้
US_INDICES = {
    "^GSPC": "S&P 500",
    "^IXIC": "Nasdaq Composite",
    "^DJI": "Dow Jones Industrial Average",
    "^RUT": "Russell 2000",
}

Session = tuple[date, float]  # (วันซื้อขาย, ราคาปิด)


def latest_closed_sessions(closes: pd.Series, cutoff: datetime) -> tuple[Session, Session] | None:
    """คืน (วันล่าสุด, วันก่อนหน้า) ที่ตลาดปิดแล้วก่อน cutoff หรือ None ถ้ามีไม่ถึง 2 วัน

    ฟังก์ชันนี้ไม่เรียกอินเทอร์เน็ต จึงทดสอบด้วยข้อมูลสมมติได้ (ดู Cell 4c)
    """
    if cutoff.tzinfo is None:
        raise ValueError("cutoff ต้องระบุเขตเวลา (timezone-aware)")
    closed = [
        (ts.date(), float(value))
        for ts, value in closes.dropna().sort_index().items()
        # ts.date() เป็นวันที่ตามเวลานิวยอร์ก ต่อกับ 16:00 นิวยอร์ก แล้วเทียบกับ cutoff (เวลาไทย) ได้โดยตรง
        if datetime.combine(ts.date(), US_MARKET_CLOSE, tzinfo=US_MARKET_TZ) <= cutoff
    ]
    if len(closed) < 2:
        return None
    return closed[-1], closed[-2]


def build_index_row(symbol: str, name: str, latest: Session, previous: Session) -> dict:
    """คำนวณการเปลี่ยนแปลงเป็นจุดและ % แล้วจัดเป็นแถวผลลัพธ์"""
    (day, close), (prev_day, prev_close) = latest, previous
    change = close - prev_close
    return {
        "symbol": symbol,
        "name": name,
        "trading_date": day.isoformat(),
        "close": round(close, 2),
        "previous_date": prev_day.isoformat(),
        "previous_close": round(prev_close, 2),
        "change_points": round(change, 2),
        "change_percent": round(change / prev_close * 100, 2),
    }


def fetch_close_history(symbol: str, start: date, end: date) -> pd.Series:
    """ดึงราคาปิดรายวันจาก Yahoo Finance (ฟังก์ชันเดียวที่เรียกอินเทอร์เน็ต)"""
    history = yf.Ticker(symbol).history(start=start, end=end, interval="1d", auto_adjust=False)
    return history["Close"] if not history.empty else pd.Series(dtype=float)


def fetch_indices(
    report_day: date,
    cutoff: datetime,
    indices: dict[str, str] | None = None,
    fetch_history: Callable[[str, date, date], pd.Series] = fetch_close_history,
) -> dict:
    """ดึงดัชนีทุกตัว ตรวจความถูกต้อง แล้วคืนผลในรูป {source, trading_date, rows, errors}

    fetch_history รับเป็นพารามิเตอร์ ตอนทดสอบจึงส่งฟังก์ชันข้อมูลสมมติเข้ามาแทนการเรียกอินเทอร์เน็ตได้
    """
    indices = indices or US_INDICES
    start = report_day - timedelta(days=HISTORY_DAYS)
    end = report_day + timedelta(days=1)  # end ไม่รวมวันนั้น จึงบวก 1 วัน
    rows, errors = [], []

    for symbol, name in indices.items():
        try:
            closes = fetch_history(symbol, start, end)
        except Exception as exc:
            errors.append(f"{name} ({symbol}): ดึงข้อมูลไม่สำเร็จ {type(exc).__name__}: {str(exc)[:200]}")
            continue
        if closes.empty:
            errors.append(f"{name} ({symbol}): ไม่ได้รับข้อมูลราคา")
            continue
        sessions = latest_closed_sessions(closes, cutoff)
        if sessions is None:
            errors.append(f"{name} ({symbol}): มีวันซื้อขายที่ปิดก่อน cutoff ไม่ถึง 2 วัน")
            continue
        rows.append(build_index_row(symbol, name, *sessions))

    # ทุกดัชนีต้องเป็นวันซื้อขายเดียวกัน ถ้าไม่ตรงกันให้บันทึกเป็น error ไม่เลือกวันให้เอง
    trading_dates = sorted({row["trading_date"] for row in rows})
    if len(trading_dates) > 1:
        errors.append(f"ดัชนีแต่ละตัวมีวันซื้อขายไม่ตรงกัน: {trading_dates}")

    return {
        "source": "Yahoo Finance (yfinance)",
        "trading_date": trading_dates[0] if len(trading_dates) == 1 else None,
        "rows": rows,
        "errors": errors,
    }

In [ ]:
# Cell 4b — เรียกใช้และแสดงผล (Cell นี้อยู่ใน Notebook ต่อ ไม่ย้ายไป .py)
us_indices_result = fetch_indices(report_day, cutoff)

statuses.append({"page": "ดัชนีหลัก US (yfinance)", "http_status": None,
                 "extracted_rows": len(us_indices_result["rows"]), "url": "https://finance.yahoo.com",
                 "note": "; ".join(us_indices_result["errors"])})
print("วันซื้อขายที่เลือก (ตามเวลานิวยอร์ก):", us_indices_result["trading_date"] or "ไม่พบ หรือวันที่ไม่ตรงกัน")
if us_indices_result["rows"]:
    display(pd.DataFrame(us_indices_result["rows"]))
for error in us_indices_result["errors"]:
    print("ข้อผิดพลาด:", error)

In [ ]:
# Cell 4c — ตรวจ logic ของ Cell 4a ด้วยข้อมูลสมมติ (ไม่ใช้อินเทอร์เน็ต)
# ถ้ามีกรณีใดผิด จะเกิด AssertionError พร้อมบอกว่ากรณีไหน
# ตอนแยกเป็น .py ส่วนนี้จะกลายเป็นไฟล์ tests/test_indices.py ที่รันด้วย pytest
BKK = ZoneInfo("Asia/Bangkok")

# ราคาปิดสมมติ: พุธ 18, พฤหัส 19, ศุกร์ 20 พ.ย. 2026 (เวลานิวยอร์ก แบบเดียวกับที่ yfinance ส่งกลับมา)
fake_closes = pd.Series(
    [100.0, 101.0, 99.0],
    index=pd.DatetimeIndex(["2026-11-18", "2026-11-19", "2026-11-20"]).tz_localize("America/New_York"),
)

# กรณี 1: รายงานเช้าวันจันทร์ → ต้องได้ข้อมูลวันศุกร์ เทียบกับวันพฤหัส (ข้ามเสาร์-อาทิตย์)
monday = datetime(2026, 11, 23, 8, 30, tzinfo=BKK)
assert latest_closed_sessions(fake_closes, monday) == ((date(2026, 11, 20), 99.0), (date(2026, 11, 19), 101.0)), "กรณี 1"

# กรณี 2: รายงานเช้าวันศุกร์ 08:30 ไทย = คืนวันพฤหัส 20:30 นิวยอร์ก → แท่งวันศุกร์ยังไม่เกิด ต้องได้วันพฤหัส
friday = datetime(2026, 11, 20, 8, 30, tzinfo=BKK)
assert latest_closed_sessions(fake_closes, friday)[0] == (date(2026, 11, 19), 101.0), "กรณี 2"

# กรณี 3: มีวันที่ปิดแล้วแค่วันเดียว → คำนวณการเปลี่ยนแปลงไม่ได้ ต้องคืน None ไม่เดาค่า
thursday = datetime(2026, 11, 19, 8, 30, tzinfo=BKK)
assert latest_closed_sessions(fake_closes, thursday) is None, "กรณี 3"

# กรณี 4: cutoff ไม่มีเขตเวลา → ต้องแจ้ง error ทันที ไม่เทียบเวลาผิดโดยไม่รู้ตัว
try:
    latest_closed_sessions(fake_closes, datetime(2026, 11, 23, 8, 30))
    raise AssertionError("กรณี 4: ควรเกิด ValueError")
except ValueError:
    pass

# กรณี 5: คำนวณการเปลี่ยนแปลง 101 → 99 = -2 จุด = -1.98%
row = build_index_row("^TEST", "Test", (date(2026, 11, 20), 99.0), (date(2026, 11, 19), 101.0))
assert (row["change_points"], row["change_percent"]) == (-2.0, -1.98), "กรณี 5"

# กรณี 6: ดัชนีหนึ่งดึงไม่สำเร็จ → อีกตัวยังได้ผล และข้อผิดพลาดถูกบันทึกใน errors
def fake_fetch(symbol, start, end):
    if symbol == "^FAIL":
        raise ConnectionError("network down")
    return fake_closes

result = fetch_indices(date(2026, 11, 23), monday, indices={"^OK": "OK", "^FAIL": "Fail"}, fetch_history=fake_fetch)
assert len(result["rows"]) == 1 and len(result["errors"]) == 1, "กรณี 6"
assert result["trading_date"] == "2026-11-20", "กรณี 6: trading_date"

print("✅ ผ่านทั้ง 6 กรณี")

## ปฏิทินเศรษฐกิจ

In [ ]:
# Cell 5 — ดึงปฏิทินเศรษฐกิจจาก Forex Factory ทั้งผลประกาศที่ผ่านมาและเหตุการณ์ที่กำลังจะเกิดขึ้น
# เวลาที่แสดงอิงเขตเวลาของเว็บไซต์ ต้องแปลงเขตเวลาก่อนนำไปเทียบกับเวลาไทย
def ff_date(day):
    return day.strftime("%b").lower() + day.strftime("%d.%Y").lstrip("0")

range_value = ff_date(calendar_start) + "-" + ff_date(calendar_end)
forex_page = get_page(
    "ปฏิทิน Forex Factory",
    "https://www.forexfactory.com/calendar",
    params={"range": range_value},
)
calendar_rows = []
if forex_page["ok"]:
    day_label = ""
    for tr in forex_page["soup"].select("tr.calendar__row"):
        date_cell = tr.select_one("td.calendar__date, .calendar__date")
        if date_cell and date_cell.get_text(" ", strip=True):
            day_label = date_cell.get_text(" ", strip=True)
        def field(selector):
            cell = tr.select_one(selector)
            return cell.get_text(" ", strip=True) if cell else ""
        event_cell = tr.select_one(".calendar__event")
        event = event_cell.get_text(" ", strip=True) if event_cell else ""
        if not event:
            continue
        link = event_cell.find("a", href=True) if event_cell else None
        impact_cell = tr.select_one(".calendar__impact span, .calendar__impact")
        calendar_rows.append({
            "date_on_page": day_label,
            "time_on_page": field(".calendar__time"),
            "currency": field(".calendar__currency"),
            "impact": (impact_cell.get("title") or " ".join(impact_cell.get("class", []))) if impact_cell else "",
            "event": event, "actual": field(".calendar__actual"),
            "forecast": field(".calendar__forecast"),
            "previous": field(".calendar__previous"),
            "event_link": urljoin(forex_page["url"], link["href"]) if link else forex_page["url"],
        })
show_page("ปฏิทิน Forex Factory", forex_page, calendar_rows,
          "เวลาที่แสดงยังเป็นเวลาดิบจากหน้าเว็บ ต้องตรวจเขตเวลาก่อนแยกเหตุการณ์อดีตหรืออนาคต")
print("ช่วงวันที่ที่ขอจากปฏิทิน:", range_value)
if calendar_rows:
    print("จำนวนเหตุการณ์ที่มีค่า Actual:", sum(bool(r["actual"]) for r in calendar_rows))
    print("จำนวนเหตุการณ์ที่ยังไม่มีค่า Actual:", sum(not r["actual"] for r in calendar_rows))

forex_factory_result = {
    "source": "Forex Factory Calendar",
    "url": forex_page["url"],
    "range": range_value,
    "rows": calendar_rows,
    "errors": [] if calendar_rows else [forex_page["error"] or "ไม่พบรายการในปฏิทิน Forex Factory"],
}

## บันทึก JSON

รวมผลจากทุกแหล่งแล้วเขียนลงไฟล์ JSON หาก `errors` เป็นรายการว่าง (`[]`) แปลว่าดึงและตรวจข้อมูลผ่านทั้งหมด

In [ ]:
# Cell 6 — รวมผลจากทุกแหล่งแล้วบันทึก JSON
# เมื่อเพิ่มแหล่งข้อมูล US ใหม่ ให้เพิ่มเข้าไปใน SOURCES ด้านล่าง
OUTPUT_FILE = Path("us_web_scraping_result.json").resolve()

SOURCES = {
    "us_indices": us_indices_result,
    "forex_factory": forex_factory_result,
}

SCRAPING_RESULT = {
    "scraped_at": datetime.now(timezone.utc).isoformat(),
    "report_date": report_day.isoformat(),
    **SOURCES,
    "errors": [f"{key}: {error}" for key, source in SOURCES.items() for error in source.get("errors", [])],
}

OUTPUT_FILE.write_text(json.dumps(SCRAPING_RESULT, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.DataFrame(statuses))
print("errors:", SCRAPING_RESULT["errors"] or "ไม่มี")
print(f"บันทึก JSON แล้ว: {OUTPUT_FILE}")